In [2]:
import duckdb
from pathlib import Path

def logical_compare_parquet(dir1, dir2):
    dir1 = str(Path(dir1))
    dir2 = str(Path(dir2))
    con = duckdb.connect()

    # List .parquet files in both dirs
    names1 = {Path(p).name for (p,) in con.execute(f"SELECT * FROM glob('{dir1}/*.parquet')").fetchall()}
    names2 = {Path(p).name for (p,) in con.execute(f"SELECT * FROM glob('{dir2}/*.parquet')").fetchall()}
    common = sorted(names1 & names2)

    only1 = sorted(names1 - names2)
    only2 = sorted(names2 - names1)
    print(f"Only in {dir1}: {only1}")
    print(f"Only in {dir2}: {only2}")

    all_ok = True
    for name in common:
        path1 = f"{dir1}/{name}"
        path2 = f"{dir2}/{name}"

        # Read schemas
        s1 = con.execute(f"DESCRIBE SELECT * FROM parquet_scan('{path1}')").fetchdf()
        s2 = con.execute(f"DESCRIBE SELECT * FROM parquet_scan('{path2}')").fetchdf()
        schema_equal = s1[['column_name','column_type']].equals(s2[['column_name','column_type']])

        # Row counts
        n1 = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{path1}')").fetchone()[0]
        n2 = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{path2}')").fetchone()[0]

        # Set difference both ways (assumes rows are comparable; if there’s no stable key,
        # this treats the entire row as a tuple)
        diff12 = con.execute(
            f"SELECT * FROM parquet_scan('{path1}') EXCEPT ALL SELECT * FROM parquet_scan('{path2}')"
        ).fetchone()
        diff21 = con.execute(
            f"SELECT * FROM parquet_scan('{path2}') EXCEPT ALL SELECT * FROM parquet_scan('{path1}')"
        ).fetchone()

        # If EXCEPT returns at least one row, DuckDB returns a row; we only need to know emptiness.
        has_diff12 = diff12 is not None
        has_diff21 = diff21 is not None

        if schema_equal and n1 == n2 and not has_diff12 and not has_diff21:
            print(f"[OK]   {name}: same schema, same rows ({n1} rows).")
        else:
            all_ok = False
            print(f"[DIFF] {name}: schema_equal={schema_equal}, rowsA={n1}, rowsB={n2}, "
                  f"A\\B_diff={has_diff12}, B\\A_diff={has_diff21}")

    if all_ok and not only1 and not only2:
        print("✅ All matching files have identical table contents.")
    else:
        print("❗Content differences found (see above).")

# Example:
path_a = "/shared/share_mala/zj2398/mimic/models/motor_8k/labels"
path_b = "/shared/share_mala/zj2398/mimic/models/mtpp_8k/labels"
logical_compare_parquet(path_a,path_b)


Only in /shared/share_mala/zj2398/mimic/models/motor_8k/labels: []
Only in /shared/share_mala/zj2398/mimic/models/mtpp_8k/labels: []
[OK]   celiac.parquet: same schema, same rows (214715 rows).
[OK]   in_hospital_mortality.parquet: same schema, same rows (331592 rows).
[OK]   long_los.parquet: same schema, same rows (331593 rows).
[OK]   masld.parquet: same schema, same rows (209133 rows).
[OK]   readmission.parquet: same schema, same rows (418850 rows).
[OK]   stroke.parquet: same schema, same rows (314140 rows).
✅ All matching files have identical table contents.


In [8]:
import pandas as pd
from pathlib import Path
base_dir = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_0.1")
# train_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform/data/train")
# for i in range(0,10):
#     train_cohort = pd.read_parquet(train_path/f"{i}.parquet")
#     print(train_cohort["subject_id"])
# meta_path = base_dir/"metadata"/"subject_splits.parquet"
# meta_path = base_dir/"metadata"/"vocabulary_cache"/"mapping"/"data.parquet"
id_list = []
data_dir = base_dir/"data"
for subdir in data_dir.iterdir():
    if not subdir.is_dir():
        continue
    for f in subdir.glob("*.parquet"):
        raw_file = pd.read_parquet(f)
        id_list.extend(raw_file["subject_id"].unique())
        # break
        # id_list.extend(raw_file)
    # break
print(len(id_list))
# meta_data = pd.read_parquet(meta_path)
# print(meta_data)

690298


In [15]:
meta_path = base_dir/"metadata"/"subject_splits.parquet"
meta_data = pd.read_parquet(meta_path)
print(meta_data)
# meta_data = meta_data[meta_data["subject_id"].isin(id_list)]
# meta_data.to_parquet(base_dir/"metadata"/"subject_splits.parquet")
# print(meta_data[meta_data["split"]=="held_out"])
# print(meta_data)

         subject_id     split
6           7046901     train
8           2007088     train
17         13628180     train
23          5935403     train
40          1538834     train
...             ...       ...
6716878     3427667  held_out
6716901    14267711  held_out
6716905     7823393  held_out
6716914      265077  held_out
6716929    12580780  held_out

[690298 rows x 2 columns]


In [2]:
from pathlib import Path
import meds_reader
import femr.splits
import pandas as pd

meds_reader_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_meds_reader")
subject_splits_path = meds_reader_path / "metadata/subject_splits.parquet"
code_metadata_path = meds_reader_path / "metadata/codes.parquet"
with meds_reader.SubjectDatabase(str(meds_reader_path), num_threads=16) as database:
    subject_ids = [_ for _ in database]
    subject_splits = pd.read_parquet(subject_splits_path)
    subject_splits = subject_splits[subject_splits.subject_id.isin(subject_ids)]
    train_tuning_split = subject_splits[~subject_splits.split.isin(["held_out"])].subject_id.tolist()
    test_split = subject_splits[subject_splits.split.isin(["held_out"])].subject_id.tolist()
    main_split = femr.splits.SubjectSplit(train_tuning_split, test_split)
    main_split.save_to_csv('/data2/processed_datasets/zj2398/femr/cumc/cumc_split.csv')

/data/anaconda3/envs/motor_chao/lib/python3.11/site-packages/meds_reader/__init__.py:205: UserWarning: Warning: You are using meds_reader multiprocessing within a Jupyter notebook. This does not reliably work and your code might randomly fail.
  warnings.warn("Warning: You are using meds_reader multiprocessing within a Jupyter notebook. This does not reliably work and your code might randomly fail.")
